# Standardize pollinator dataset to Camtrap DP (Insect AI extension)

Converts the filtered camera trap data (images in `media/`, Pascal VOC XMLs in `raw-data/`) into Camtrap DP tables.

Prerequisite: run `filter_images.ipynb` in the project root first to populate `media/` and `raw-data/`.

In [6]:
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS

raw_data = Path("../raw-data")
media_dir = Path("../media")
output_dir = Path("..")

def get_exif(img_path):
    """Return EXIF as {tag_name: value} dict, or empty dict."""
    try:
        exif = Image.open(img_path)._getexif()
        if not exif:
            return {}
        return {TAGS.get(k, k): v for k, v in exif.items()}
    except Exception:
        return {}

# Pair each XML with its image in media/
target_images = []
for xml_file in sorted(raw_data.glob("*.xml")):
    stem = xml_file.stem
    for ext in [".jpg", ".JPG"]:
        img = media_dir / f"{stem}{ext}"
        if img.exists():
            target_images.append((stem, img, xml_file))
            break

print(f"Camera trap images: {len(target_images)}")

Camera trap images: 256


## Step 1: Build media table

Build `media.csv` according to the [Camtrap DP media spec](https://camtrap-dp.tdwg.org/data/#media).

Timestamps come from EXIF `DateTimeOriginal`. No timezone offset is recorded in EXIF, but the cameras were deployed in the Balearic Islands (Europe/Madrid timezone), so we localize accordingly.

In [7]:
import pandas as pd
import json
from datetime import datetime
from zoneinfo import ZoneInfo

TZ = ZoneInfo("Europe/Madrid")

rows = []
for stem, img_path, xml_path in target_images:
    exif = get_exif(img_path)

    # Parse timestamp from EXIF and localize
    dt_str = exif.get("DateTimeOriginal", "")
    if dt_str:
        dt_naive = datetime.strptime(dt_str, "%Y:%m:%d %H:%M:%S")
        dt_local = dt_naive.replace(tzinfo=TZ)
        timestamp = dt_local.isoformat()
    else:
        timestamp = None

    # Collect EXIF data as JSON
    exif_json = json.dumps({
        k: v for k, v in exif.items()
        if isinstance(v, (str, int, float, bool))
    })

    rows.append({
        "mediaID": stem,
        "deploymentID": None,  # to be filled in deployments step
        "captureMethod": "timeLapse",
        "timestamp": timestamp,
        "filePath": f"media/{img_path.name}",
        "filePublic": True,
        "fileName": img_path.name,
        "fileMediatype": "image/jpeg",
        "exifData": exif_json,
        "favorite": None,
        "mediaComments": None,
    })

media = pd.DataFrame(rows)
print(f"{len(media)} media records")
print(f"Timestamp range: {media['timestamp'].min()} to {media['timestamp'].max()}")
media.head()

256 media records
Timestamp range: 2020-02-13T12:51:09+01:00 to 2021-05-30T10:47:14+02:00


,mediaID,deploymentID,captureMethod,timestamp,filePath,filePublic,fileName,fileMediatype,exifData,favorite,mediaComments
0,5mp_010421_17_24_24_049067,None,timeLapse,2021-04-01T17:24:24+02:00,media/5mp_010421_17_24_24_049067.jpg,True,5mp_010421_17_24_24_049067.jpg,image/jpeg,"{""ImageWidth"": 1280, ""ImageLength"": 960, ""Reso...",None,None
1,5mp_010421_17_24_35_519878,None,timeLapse,2021-04-01T17:24:36+02:00,media/5mp_010421_17_24_35_519878.jpg,True,5mp_010421_17_24_35_519878.jpg,image/jpeg,"{""ImageWidth"": 1280, ""ImageLength"": 960, ""Reso...",None,None
2,5mp_010421_17_24_36_834534,None,timeLapse,2021-04-01T17:24:37+02:00,media/5mp_010421_17_24_36_834534.jpg,True,5mp_010421_17_24_36_834534.jpg,image/jpeg,"{""ImageWidth"": 1280, ""ImageLength"": 960, ""Reso...",None,None
3,5mp_010421_17_24_41_289453,None,timeLapse,2021-04-01T17:24:41+02:00,media/5mp_010421_17_24_41_289453.jpg,True,5mp_010421_17_24_41_289453.jpg,image/jpeg,"{""ImageWidth"": 1280, ""ImageLength"": 960, ""Reso...",None,None
4,5mp_010421_17_24_42_529702,None,timeLapse,2021-04-01T17:24:43+02:00,media/5mp_010421_17_24_42_529702.jpg,True,5mp_010421_17_24_42_529702.jpg,image/jpeg,"{""ImageWidth"": 1280, ""ImageLength"": 960, ""Reso...",None,None


In [8]:
media.to_csv(output_dir / "media.csv", index=False)
print("Saved media.csv")

Saved media.csv


## Step 2: Build observations table

Build `observations.csv` from the Pascal VOC XML annotations. Each `<object>` becomes one observation row, linked to the media record via `mediaID`. Bounding boxes are normalized to 0–1 range as required by the spec.

In [9]:
import xml.etree.ElementTree as ET

obs_rows = []
obs_counter = 0

for stem, img_path, xml_path in target_images:
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Image dimensions for normalizing bounding boxes
    img_w = int(root.find(".//size/width").text)
    img_h = int(root.find(".//size/height").text)

    # Get the timestamp from the media table for eventStart/eventEnd
    media_row = media.loc[media["mediaID"] == stem].iloc[0]
    timestamp = media_row["timestamp"]

    for obj in root.findall("object"):
        obs_counter += 1
        name = obj.find("name").text

        bb = obj.find("bndbox")
        xmin = int(bb.find("xmin").text)
        ymin = int(bb.find("ymin").text)
        xmax = int(bb.find("xmax").text)
        ymax = int(bb.find("ymax").text)

        obs_rows.append({
            "observationID": f"obs{obs_counter:04d}",
            "deploymentID": None,  # to be filled in deployments step
            "mediaID": stem,
            "eventID": None,
            "eventStart": timestamp,
            "eventEnd": timestamp,
            "observationLevel": "media",
            "observationType": "animal",
            "cameraSetupType": None,
            "scientificName": name,
            "count": 1,
            "lifeStage": None,
            "sex": None,
            "behavior": None,
            "individualID": None,
            "individualPositionRadius": None,
            "individualPositionAngle": None,
            "individualSpeed": None,
            "bboxX": round(xmin / img_w, 6),
            "bboxY": round(ymin / img_h, 6),
            "bboxWidth": round((xmax - xmin) / img_w, 6),
            "bboxHeight": round((ymax - ymin) / img_h, 6),
            "classificationMethod": "human",
            "classifiedBy": "Pau Enric Serra Marin",
            "classificationTimestamp": None,
            "classificationProbability": None,
            "observationTags": None,
            "observationComments": None,
        })

observations = pd.DataFrame(obs_rows)
print(f"{len(observations)} observations from {observations['mediaID'].nunique()} media files")
print(f"\nTop taxa:")
print(observations["scientificName"].value_counts().head(10))
observations.head()

309 observations from 256 media files

Top taxa:
scientificName
Apis mellifera         82
Vanessa cardui         40
Hoverfly               38
Unidentified insect    35
Butterfly              28
bee                    22
Beetle                 18
Eucera                 18
Fly                    15
Bombus terrestris       6
Name: count, dtype: int64


,observationID,deploymentID,mediaID,eventID,eventStart,eventEnd,observationLevel,observationType,cameraSetupType,scientificName,...,bboxX,bboxY,bboxWidth,bboxHeight,classificationMethod,classifiedBy,classificationTimestamp,classificationProbability,observationTags,observationComments
0,obs0001,None,5mp_010421_17_24_24_049067,None,2021-04-01T17:24:24+02:00,2021-04-01T17:24:24+02:00,media,animal,None,Vanessa cardui,...,0.489844,0.280208,0.121094,0.129167,human,Pau Enric Serra Marin,None,None,None,None
1,obs0002,None,5mp_010421_17_24_35_519878,None,2021-04-01T17:24:36+02:00,2021-04-01T17:24:36+02:00,media,animal,None,Vanessa cardui,...,0.487500,0.185417,0.103906,0.154167,human,Pau Enric Serra Marin,None,None,None,None
2,obs0003,None,5mp_010421_17_24_36_834534,None,2021-04-01T17:24:37+02:00,2021-04-01T17:24:37+02:00,media,animal,None,Vanessa cardui,...,0.468750,0.211458,0.115625,0.112500,human,Pau Enric Serra Marin,None,None,None,None
3,obs0004,None,5mp_010421_17_24_41_289453,None,2021-04-01T17:24:41+02:00,2021-04-01T17:24:41+02:00,media,animal,None,Vanessa cardui,...,0.493750,0.194792,0.110937,0.143750,human,Pau Enric Serra Marin,None,None,None,None
4,obs0005,None,5mp_010421_17_24_42_529702,None,2021-04-01T17:24:43+02:00,2021-04-01T17:24:43+02:00,media,animal,None,Vanessa cardui,...,0.482812,0.218750,0.073438,0.105208,human,Pau Enric Serra Marin,None,None,None,None


In [10]:
observations.to_csv(output_dir / "observations.csv", index=False)
print("Saved observations.csv")

Saved observations.csv
